In [ ]:
import os
import pytest
from pytest_html import extras
from datetime import date, timedelta
import openpyxl
import base64

# Function to add command line option for selecting tests
def pytest_addoption(parser):
    parser.addoption(
        "--tests", action="store", default="", help="Comma-separated list of tests to run"
    )

# Fixture to retrieve selected tests from command line option or file
@pytest.fixture
def selected_tests(request):
    tests = request.config.getoption("--tests")
    print("Selected tests:", tests)
    return tests.split(",") if tests else []

# Modify pytest collection to include only selected tests
def pytest_collection_modifyitems(config, items):
    selected_tests = config.getoption("--tests")
    selected_tests = selected_tests.split(",") if selected_tests else read_test_selection_file()

    if selected_tests:
        selected_tests_set = set(selected_tests)
        keep_items = []
        for item in items:
            for mark in item.iter_markers():
                if mark.name in selected_tests_set:
                    keep_items.append(item)
                    break
        items[:] = keep_items

# Customize HTML report title with yesterday's date
def pytest_html_report_title(report):
    yesterday = date.today() - timedelta(days=1)
    title = "UAT Testing Report: " + yesterday.strftime("%m/%d/%Y")
    report.title = title

# Configure pytest for HTML report generation
@pytest.hookimpl(tryfirst=True)
def pytest_configure(config):
    if config.getoption("--html"):
        yesterday = date.today() - timedelta(days=1)
        title = "UAT_Testing_" + yesterday.strftime("%m_%d_%Y")
        report_filename = f"{title}.html"
        config.option.htmlpath = report_filename
        config.addinivalue_line("markers", "custom_info: add custom info to the html report")

    # Create an Excel workbook and add a sheet for the report
    config._excel_workbook = openpyxl.Workbook()
    config._excel_sheet = config._excel_workbook.active
    config._excel_sheet.title = "Test Results"
    headers = ["Test Name", "Case Name", "Pass/Fail", "Value DB1", "Value DB2", "Additional Info 1", "Additional Info 2", "Additional Info 3", "Additional Info 4"]
    config._excel_sheet.append(headers)

    # Save the Excel file
    excel_filename = f"UAT_Testing_Report_{yesterday.strftime('%m_%d_%Y')}.xlsx"
    config._excel_filename = excel_filename
    config._excel_workbook.save(excel_filename)

# Fixture to track failed cases for each test
@pytest.fixture
def track_failed_cases(request):
    request.node.list_failed = []
    request.node.config = request.config
    return request.node.list_failed

# Function to add a failed case with details from two databases
def add_case_result(track_failed_cases, case_name, value_db1, value_db2, pass_fail_flag, additional_info):
    case_detail = (case_name, value_db1, value_db2, pass_fail_flag, *additional_info)
    track_failed_cases.append(case_detail)

    # Write the case detail to the Excel sheet
    request = track_failed_cases.request
    sheet = request.config._excel_sheet
    test_name = request.node.name
    row = [test_name, case_name, pass_fail_flag, value_db1, value_db2, *additional_info]
    sheet.append(row)
    request.config._excel_workbook.save(request.config._excel_filename)

# Hook to add extra HTML info for failed cases to the report
@pytest.hookimpl(hookwrapper=True)
def pytest_runtest_makereport(item, call):
    outcome = yield
    rep = outcome.get_result()
    if rep.when == "call" and hasattr(item, 'list_failed'):
        extra = getattr(rep, "extra", [])
        
        # Calculate the number of failed and passed cases
        failed_cases = [case for case in item.list_failed if case[3] == "FAIL"]
        passed_cases = [case for case in item.list_failed if case[3] == "PASS"]

        # Format the failed cases in red and bold
        info = "<p><strong>Failed Cases:</strong></p><ul>"
        for case in failed_cases:
            case_name, value_db1, value_db2, _, *additional_info = case
            info += f"<li><strong style='color: red;'>{case_name}: Value in DB1 - {value_db1}, Value in DB2 - {value_db2}, Additional Info - {additional_info}</strong></li>"
        info += "</ul>"

        # Format the passed cases and handle truncation
        info += "<p><strong>Passed Cases:</strong></p><ul>"
        if len(passed_cases) > 20:
            truncated_passed_cases = passed_cases[:20]
            for case in truncated_passed_cases:
                case_name, value_db1, value_db2, _, *additional_info = case
                info += f"<li>{case_name}: Value in DB1 - {value_db1}, Value in DB2 - {value_db2}, Additional Info - {additional_info}</li>"
            info += f"<li><strong>List truncated. 20 rows of {len(passed_cases)} shown</strong></li>"
        else:
            for case in passed_cases:
                case_name, value_db1, value_db2, _, *additional_info = case
                info += f"<li>{case_name}: Value in DB1 - {value_db1}, Value in DB2 - {value_db2}, Additional Info - {additional_info}</li>"
        info += "</ul>"

        extra.append(extras.html(info))
        rep.extra = extra

# Function to read selected test marks from a file
def read_test_selection_file():
    selected_tests = []
    if os.path.exists("test_selection.txt"):
        with open("test_selection.txt", "r") as file:
            for line in file:
                selected_tests.append(line.strip())
    return selected_tests

# Function to run selected tests based on marks
def run_selected_tests(selected_tests):
    markers = " or ".join(selected_tests)
    pytest_args = ["-v", "-m", markers, "--html=report.html"]
    pytest.main(pytest_args)

# Hook to add description to each test in the HTML report
@pytest.hookimpl(hookwrapper=True)
def pytest_html_results_table_row(report, cells):
    outcome = yield
    description = None
    if hasattr(report, 'user_properties'):
        for name, value in report.user_properties:
            if name == 'description':
                description = value
                break
    cells.insert(2, f"<td>{description or 'N/A'}</td>")

# Hook to add an extra column to the results table header for descriptions
@pytest.hookimpl(hookwrapper=True)
def pytest_html_results_table_header(cells):
    outcome = yield
    cells.insert(2, "<th>Description</th>")

# Hook to add a link to the Excel report in the HTML report
def pytest_sessionfinish(session, exitstatus):
    excel_filename = session.config._excel_filename
    with open(excel_filename, "rb") as f:
        data = f.read()
    b64_data = base64.b64encode(data).decode()
    html = f'<p><a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64_data}" download="{excel_filename}">Download Excel Report</a></p>'
    session.config._metadata["Excel Report"] = html
